# Overview

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kootru-repo/charge-filtered-cumulant-residuals/blob/main/notebooks/00_overview.ipynb)


Reproducibility envelope for the manuscript:

> **Charge- and block-refined bias bounds for second-order cumulant truncation on $\mathrm{U}(1)$-invariant fermionic states.**

## How to use this repo

Each numbered notebook below verifies a specific group of manuscript claims. Each ends with explicit `assert` cells anyone can run to confirm the headline result of that section.

| # | Notebook | Manuscript section | Headline claim |
|---|---|---|---|
| 01 | `01_partition_constants.ipynb` | Sec III, Theorem 1 | $M_3=6$, $M_4=26$, $M_5=150$, $B_4=105$, $B_5=227\,251$ |
| 02 | `02_chemistry_catalog.ipynb` | Sec III, Cor 1 | Block-refined $\widehat B^{\mathrm{charge}}_4(W) \in \{1, 3, 5\}$ on the chemistry catalog |
| 03 | `03_implementation_audit.ipynb` | Sec VI | $3679$-observable audit, $26$ fixed-$N$ states, $B^{\mathrm{eff}}_{\max} \approx 2.0$ |
| 04 | `04_correlated_calibration.ipynb` | Sec V baseline + correlated-state extension | $\Delta^{\mathrm{cat}}=0$ at $U=0$ Hubbard; grows with $U/t$ |
| 05 | `05_diagnostic_ucb_demo.ipynb` | Sec IV | Sample-split UCB on synthetic shadows; one-sided coverage |

## Two reproduction paths

1. **Run notebooks here**: install with `uv sync --extra dev --extra notebooks`, then open each in order with `uv run jupyter lab` and click "Run All". Every notebook ends with `assert` cells.
2. **Run the test suite locally**: `uv sync --extra dev && uv run pytest`. ~30 unit tests, ~2 minutes.
3. **Verify deposited data only**: `uv run pytest tests/test_data_integrity.py`. Hashes the JSON outputs in `data/` against `MANIFEST.json` SHA256 entries.
4. **Open any notebook on Colab**: click the badge at the top of each notebook. The bootstrap cell clones the repo to `/content`, installs the package via uv, and the rest of the notebook runs unchanged.

## What's NOT in this repo

- The manuscript LaTeX source, supplements, cover letter (a separate, manuscript-focused project).
- Hardware experiment data (none in this manuscript version).
- Development history (intermediate drafts, design notes); irrelevant to reproducibility.
- The full $3679$-observable audit re-run on a free-tier CPU (intractable; notebook 03 runs a representative subset and loads the deposited JSON for headline numbers).
- H$_2$ / LiH / STO-3G FCI cells of the calibration plan, deferred until PySCF availability.

In [ ]:
# Bootstrap on Colab / fresh env (skipped if package + data are already present).
# Local users who cloned the repo and ran 'uv sync' skip the install entirely.
# On Colab the bootstrap clones the repo so data/ files are reachable via
# notebook-relative paths, chdir's into it, then installs the package with uv.
import importlib, importlib.util as _ilu
import pathlib as _pl
_HAS_PKG = _ilu.find_spec("connected_layer_sector") is not None
_HAS_DATA = _pl.Path("data").is_dir() or _pl.Path("../data").is_dir()
if not (_HAS_PKG and _HAS_DATA):
    import subprocess as _sp, os as _os
    _REPO = "/content/charge-filtered-cumulant-residuals"
    if not _pl.Path(_REPO).is_dir():
        _sp.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/kootru-repo/charge-filtered-cumulant-residuals.git",
            _REPO,
        ])
    _os.chdir(_REPO)
    # Pin uv to a known directory. UV_INSTALL_DIR is the installer's explicit
    # override; without it the location depends on CARGO_HOME / XDG_BIN_HOME
    # inheritance (Colab differs from local Linux).
    _UV_DIR = "/tmp/uv-bootstrap"
    _os.makedirs(_UV_DIR, exist_ok=True)
    _env = {**_os.environ, "UV_INSTALL_DIR": _UV_DIR}
    _sp.check_call(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, executable="/bin/bash", env=_env,
    )
    _UV = f"{_UV_DIR}/uv"
    # Non-editable install: lands directly in site-packages (already on sys.path),
    # so the import below works without sys.path gymnastics. Editable installs
    # rely on .pth files Python only reads at startup.
    _sp.check_call([_UV, "pip", "install", "--system", "-q", "."])
    importlib.invalidate_caches()  # flush find_spec cache so re-runs idempotently short-circuit

import connected_layer_sector
print(f"connected_layer_sector {connected_layer_sector.__version__} ready")


## Quick sanity check

Verify the package imports cleanly and the manuscript's headline constants are reproduced. The next cell asserts:

1. **Partition-lattice constants** (Sec III, Thm 1): $M_3 = 6$, $M_4 = 26$, $M_5 = 150$, $B_3 = 1$, $B_4 = 105$, $B_5 = 227\,251$.
2. **Charge-filtered polynomial** worked example (Sec III): $B^{\mathrm{charge}}_4(\mathtt{a^\dagger a n n}) = 53$, $B^{\mathrm{charge}}_5(\mathtt{a^\dagger a n n}) = 301$.
3. **Block-refined chemistry catalog** (Sec III, Cor 1): $\widehat B^{\mathrm{charge}}_4(W) \in \{1, 1, 3, 1, 5\}$ across the five catalog word types.

Notebooks 01-05 below drill into each of these and add the audit, the correlated-state calibration, and the diagnostic UCB.

In [ ]:
import connected_layer_sector as cls
from connected_layer_sector import Bhat_charge_r
from connected_layer_sector.constants import B_charge_r

print(f"package version: {cls.__version__}")
print()

# Partition-lattice constants M_r and B_r (Sec III, Theorem 1).
print(f"M_3 = {cls.M_r_const(3):.0f}   (manuscript: 6)")
print(f"M_4 = {cls.M_r_const(4):.0f}  (manuscript: 26)")
print(f"M_5 = {cls.M_r_const(5):.0f} (manuscript: 150)")
print(f"B_3 = {cls.B_r_const(3):.0f}   (manuscript: 1)")
print(f"B_4 = {cls.B_r_const(4):.0f} (manuscript: 105)")
print(f"B_5 = {cls.B_r_const(5):.0f} (manuscript: 227,251)")

assert cls.M_r_const(3) == 6.0
assert cls.M_r_const(4) == 26.0
assert cls.M_r_const(5) == 150.0
assert cls.B_r_const(4) == 105.0
assert cls.B_r_const(5) == 227251.0

# Charge-filtered polynomial B^charge_r(W), worked example from Sec III.
print()
print(f"B^charge_4(a^d a n n) = {B_charge_r(1, 2, 4):.0f}  (manuscript Sec III worked example: 53)")
print(f"B^charge_5(a^d a n n) = {B_charge_r(1, 2, 5):.0f} (manuscript: 301)")

assert B_charge_r(1, 2, 4) == 53.0
assert B_charge_r(1, 2, 5) == 301.0

# Block-refined chemistry-catalog constants (Sec III, Corollary 1).
# Each row: word label, (h_pairs, z_n_letters, r), expected from manuscript.
print()
print("Bhat^charge_4 on the chemistry catalog (Sec III, Cor 1):")
catalog = [
    ("n_i n_j n_k",            (0, 3, 4), 1.0),
    ("a^d_i a_j n_k",          (1, 1, 4), 1.0),
    ("a^d_i a_j n_k n_l",      (1, 2, 4), 3.0),
    ("a^d_i a^d_j a_k a_l",    (2, 0, 4), 1.0),
    ("n_i n_j n_k n_l",        (0, 4, 4), 5.0),
]
for label, args, expected in catalog:
    val = Bhat_charge_r(*args)
    print(f"  {label:<24} = {val:.0f}  (manuscript: {expected:.0f})")
    assert val == expected, f"{label}: got {val}, expected {expected}"

print()
print("PASS: all headline partition-lattice and chemistry-catalog constants match the manuscript.")